# Загрузка PRD-модели из MLflow и тестовый предикт

Продакшен-модель (тег `deployment=PRD`) из MLflow и выполнение инференса.


## Инфраструктура
- **MLflow Tracking Server**: http://localhost:5050
- **MinIO S3 (артефакты)**: http://localhost:9000  (bucket: `mlflow-artifacts`)
- **MinIO Console**: http://localhost:9001  (login: `minioadmin` / `minioadmin`)

Запуск: 

`docker compose up -d` или `make dc.up`

## 1. Подключение к MLflow

In [21]:
import os
import pandas as pd
import mlflow.lightgbm

In [22]:
import dotenv

dotenv.load_dotenv('../../.env')

MLFLOW_URI = os.getenv('MLFLOW_URI')  # 'http://localhost:5050'
EXPERIMENT_NAME = os.getenv('EXPERIMENT_NAME')  # 'reranker-lgbm'

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow tracking URI: {mlflow.get_tracking_uri()}')
print(f'Experiment: {EXPERIMENT_NAME}')

MLflow tracking URI: http://localhost:5050
Experiment: reranker-lgbm


## 2. Поиск PRD-модели по тегу

In [23]:
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(EXPERIMENT_NAME)

prd_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.deployment = 'PRD'",
    order_by=['start_time DESC'],
    max_results=1
)

prd_run = prd_runs[0]
RUN_ID = prd_run.info.run_id



In [24]:
THRESHOLD = float(prd_run.data.params.get('decision_threshold', 0.5))

print('Threshold:', THRESHOLD)
prd_run.data.tags

Threshold: 0.3


{'mlflow.user': 'andrey',
 'mlflow.source.name': 'train_reranker_mlflow.ipynb',
 'mlflow.source.type': 'NOTEBOOK',
 'mlflow.runName': 'final_lgbm_balanced',
 'model_type': 'LGBMClassifier',
 'boosting_type': 'gbdt',
 'dataset': 'merged_for_boosting.csv',
 'features': 'max,mean,std,perc_90,num_passed,range',
 'selected_as': 'lgbm_balanced',
 'stage': 'final',
 'deployment': 'PRD',
 'prd_reason': 'Best val_auc=0.8133, test_auc=0.8634, low overfitting=0.1748'}

### Основные метрики

In [25]:
prd_run.data.metrics

{'train_auc': 0.9880977721016035,
 'train_f1': 0.8706896551724138,
 'train_accuracy': 0.9523809523809523,
 'train_precision': 0.8145161290322581,
 'train_recall': 0.9351851851851852,
 'val_auc': 0.8132763975155279,
 'val_f1': 0.55,
 'val_accuracy': 0.8666666666666667,
 'val_precision': 0.6470588235294118,
 'val_recall': 0.4782608695652174,
 'test_auc': 0.8633540372670807,
 'test_f1': 0.6122448979591837,
 'test_accuracy': 0.8592592592592593,
 'test_precision': 0.5769230769230769,
 'test_recall': 0.6521739130434783,
 'test_false_negatives': 8.0,
 'test_false_positives': 11.0,
 'robustness_auc_noise_0_01': 0.8642080745341614,
 'robustness_drop_noise_0_01': -0.0008540372670806651,
 'robustness_auc_noise_0_05': 0.8644409937888199,
 'robustness_drop_noise_0_05': -0.0010869565217391797,
 'robustness_auc_noise_0_1': 0.865527950310559,
 'robustness_drop_noise_0_1': -0.0021739130434782483}

## 3. Загрузка модели из S3 (MLflow artifacts)

In [26]:
model_uri = f'runs:/{RUN_ID}/model'
loaded_model = mlflow.lightgbm.load_model(model_uri)
print(loaded_model)

LGBMClassifier(colsample_bytree=0.9, learning_rate=0.03, metric='auc',
               min_child_samples=10, n_estimators=500, objective='binary',
               random_state=42, scale_pos_weight=np.float64(4.833333333333333),
               subsample=0.9, verbose=-1)


## 4. Тестовый предикт

Признаки: `max`, `mean`, `std`, `perc_90`, `num_passed`, `range` - полученные из работы движка

In [27]:
FEATURES = ['max', 'mean', 'std', 'perc_90', 'num_passed', 'range']

# Предикт из реальных данных (если доступен merged_for_boosting.csv)

In [17]:
real_data_path = '../../model/merged_for_boosting.csv'

df_real = pd.read_csv(real_data_path)
sample = df_real[FEATURES].sample(10, random_state=42)
real_proba = loaded_model.predict_proba(sample)[:, 1]
sample = sample.copy()
sample['y_true'] = df_real.loc[sample.index, 'rel'].values
sample['y_proba'] = real_proba.round(4)
sample['y_pred'] = (real_proba >= THRESHOLD).astype(int)

sample

,max,mean,std,perc_90,num_passed,range,y_true,y_proba,y_pred
70,0.225342,0.193150,0.015140,0.213977,120,0.074219,0,0.1266,0
827,0.241455,0.216991,0.013725,0.229370,73,0.058350,0,0.1267,0
231,0.243652,0.219427,0.012118,0.235840,51,0.050049,0,0.1414,0
588,0.250732,0.204761,0.014545,0.222778,121,0.074585,0,0.1268,0
39,0.231689,0.207316,0.016272,0.225623,24,0.052368,1,0.2963,0
731,0.274658,0.247693,0.013041,0.259106,50,0.081299,1,0.3838,1
299,0.251221,0.207411,0.013768,0.223706,113,0.078613,0,0.1864,0
110,0.256836,0.205760,0.012335,0.223047,497,0.078979,0,0.1710,0
72,0.216553,0.202898,0.013051,0.216309,21,0.035767,0,0.1269,0
86,0.226074,0.189880,0.017176,0.210229,70,0.071655,0,0.1266,0


## 5. Информация о модели из MLflow

In [11]:
print(f'Run ID: {RUN_ID}')
print(f'Experiment: {EXPERIMENT_NAME}')
print(f'Model URI: {model_uri}')

Run ID: cd5cfdc8b72f475298e4f5b8235fbddd
Experiment: reranker-lgbm
Model URI: runs:/cd5cfdc8b72f475298e4f5b8235fbddd/model


### Параметры

In [29]:
prd_run.data.params

{'seed': '42',
 'test_size': '0.15',
 'val_size': '0.15',
 'n_train': '630',
 'n_val': '135',
 'n_test': '135',
 'class_imbalance_ratio': '4.8333',
 'decision_threshold': '0.3',
 'objective': 'binary',
 'metric': 'auc',
 'boosting_type': 'gbdt',
 'num_leaves': '31',
 'min_child_samples': '10',
 'scale_pos_weight': '4.833333333333333',
 'subsample': '0.9',
 'colsample_bytree': '0.9',
 'learning_rate': '0.03',
 'n_estimators': '500',
 'random_state': '42',
 'verbose': '-1'}